# Virus simulation (grp-SIS on networks): Prototype Analysis

**Project:** Testing a spectral epidemic threshold for SIS on Erdős–Rényi networks under heavy-tailed recovery (RMAI / Open University IM1312).

This notebook follows the *Prototype / frame the problem* checklist from [Hands-On Machine Learning](https://github.com/ageron/handson-ml3) (Ch. 2), adapted to a **computational epidemiology** study (agent-based simulation + spectral reference + Python analysis) rather than a classical supervised-ML product.

**Artifacts:** `proposal/research_proposal.pdf`, `report/report.pdf`, pipeline under `scripts/`, NetLogo model in `netlogo/`.

## Summary

We study whether the mean-field spectral benchmark $\tau_c \approx 1/\lambda_{\max}$ (equivalently $\beta_{\mathrm{pred}} = 1/(\lambda_{\max}\,\mathbb{E}[W])$) still guides **when** a susceptible–infected–susceptible (grp-SIS) epidemic persists in an **agent-based** NetLogo model when recovery times are **heavy-tailed** but share the same mean $\mathbb{E}[W]=5$ ticks.

**Setup:** Erdős–Rényi contact networks ($N=2000$, mean degree $\approx 6$), three independent graph seeds (`10001`–`10003`), three recovery laws (exponential, Tang power law, Tang lognormal), fine sweeps of per-edge infection probability $\beta$, 24 BehaviorSpace replicates per $(\beta,\text{regime})$.

**Main findings (see `report/report.pdf` and `output/empirical_thresholds.csv`):**

- The empirical **survival** threshold $\hat{\beta}_{\mathrm{surv}\,50}$ (smallest $\beta$ with $\geq 50\%$ runs not extinct by `max-ticks`) lies **above** $\beta_{\mathrm{pred}}$ for all regimes: ratio $\hat{\beta}/\beta_{\mathrm{pred}} \approx 1.66$–$1.73$ across seeds (Table in `output/threshold_ratio_summary.csv`).
- Differences **between** recovery laws at threshold location are **modest** compared to the gap vs. spectral prediction; **extinction times** and variability differ more clearly (supports **RQ3**).
- The original hypothesis (heavy tails uniformly ease persistence at fixed $\mathbb{E}[W]$) is **partly qualified**: power-law recovery can cross 50% survival at slightly lower $\beta$ than exponential/lognormal on some seeds, but exponential and lognormal often **hit the upper $\beta$ grid cap** (censoring at `0.048`).

**Deliverables:** reproducible pipeline (`python scripts/run_pipeline.py --all`), analysis notebooks in `notebooks/`, written report in `report/`.

## Probleem Statement (Frame the problem and Look at the Big Picture)

#### 1. Define the objective in business terms.

**Stakeholder question (public health / epidemiology):** On a given contact structure, is an infection likely to **die out** or **keep circulating** at low prevalence—and can we trust simple rules that combine **spreading strength**, **mean infectious period**, and **network connectivity**?

**Project objective:** Quantify how far simulation outcomes deviate from the spectral mean-field line $\tau_c \approx 1/\lambda_{\max}$ when recovery is non-exponential, and whether heavy-tailed recovery shifts the **operational** persistence threshold or mainly **dynamics** (extinction times, variability).

#### 2. How will the solution be used?

- **Research:** Answer **RQ1–RQ3** in the proposal; support the master’s report and reproducible artifacts.
- **Methodological:** Provide a documented NetLogo + Python workflow others can rerun on new graph seeds or $\beta$ grids.
- **Not** a deployed real-time outbreak dashboard: outputs are offline CSVs, figures, and LaTeX macros—not a production surveillance product.

#### 3. What are the current solutions/workarounds (if any)?

- **Spectral / N-intertwined mean-field** thresholds ($\tau_c \approx c/\lambda_{\max}$, often $c=1$).
- **Continuous-time Markov SIS** on networks with exponential recovery.
- **grp-SIS theory** (Tang et al.) for general recovery-time laws—does not automatically yield the same $\lambda_{\max}$ line for heavy tails.
- **Workaround in practice:** treat the spectral line as a **benchmark** and validate with simulation when recovery is irregular.

#### 4. How should we frame the problem (supervised/unsupervised, online/offline etc.)

- **Offline computational experiment**, not predictive ML on labelled human data.
- Closest ML framing: **stochastic simulation study** with factorial design (recovery regime × $\beta$ × graph seed × replicate).
- “Target” is an **operational epidemic threshold** (e.g. $\hat{\beta}_{\mathrm{surv}\,50}$), estimated from Monte Carlo replicates—not a classifier trained on features.

#### 5. How should performance be measured?

| Quantity | Definition |
|----------|------------|
| $\beta_{\mathrm{pred}}$ | $1/(\lambda_{\max}\,\mathbb{E}[W])$ from exported edges |
| $\hat{\beta}_{\mathrm{surv}\,50}$ | Smallest $\beta$ on sweep grid with survival rate $\geq 0.5$ |
| Ratio $\hat{\beta}/\beta_{\mathrm{pred}}$ | How conservative the spectral line is for this ABM |
| Extinction time | `bs-out-final-tick` when `bs-out-extinct` = 1 |
| Late prevalence | `bs-out-late-mean-prevalence`; trajectory check in experiment `08` |

Bootstrap 95% CIs and logistic ED50 alternatives are implemented in `scripts/threshold_estimators.py`.

#### 6. Is the performance measure aligned with the business objective?

**Yes, with caveats.** Survival at a fixed horizon (`max-ticks` = 10 000) operationalises “still circulating” for RQ1–RQ2; extinction-time summaries address RQ3 (transients vs. threshold location). The horizon mixes true endemicity with slow extinction—documented as a limitation in `report/report.tex`.

#### 7. What should be the minimum performance needed to reach the business objective?

Success = **reproducible evidence** answering RQ1–RQ3 with stated uncertainty, not beating a accuracy score.

Minimum bar met in this repo:

- Spectral reference computed for each ER seed.
- Full ER baseline sweeps (`05`–`07`) aggregated to `output/empirical_thresholds.csv`.
- Clear statement when estimators are **censored** at $\beta_{\max}=0.048$.

#### 8. What are comparable problems? Can we reuse experience or tools?

- Network epidemic thresholds (Chakrabarti et al.; Pastor-Satorras et al.; Van Mieghem N-intertwined SIS).
- Non-Markovian recovery (Cator et al.; Tang et al. grp-SIS).
- **Tools reused:** NetLogo ABM, NumPy/SciPy for $\lambda_{\max}$, pandas/matplotlib/seaborn pipeline, optional fast Python benchmark in `benchmark/`.

#### 9. Is human expertise available?

- Course supervisors / RMAI context; epidemiology and network-science literature cited in `references/`.
- NetLogo and BehaviorSpace documentation for ABM design.

#### 10. How would we solve the problem manually?

1. Build ER graph, export edges, compute $\lambda_{\max}$ in Python.
2. Run many stochastic replicates per $\beta$ and recovery law.
3. Plot survival vs. $\beta$; read off 50% crossing; compare to $\beta_{\mathrm{pred}}$.
4. Inspect extinction-time distributions and prevalence trajectories—not only the threshold point.

#### 11. List the assumptions we (or others) have made so far.

- Fixed ER specification ($N$, mean degree); not a claim about real contact networks.
- $\mathbb{E}[W]=5$ ticks matched across recovery choosers via `recovery-mean`.
- Initial outbreak $I_0=5$ ($\rho_0=0.25\%$).
- Spectral line with $c=1$ is a **continuous-time mean-field** benchmark; discrete-time grp-SIS may differ.
- Unweighted simple graphs; homogeneous $\beta$ per edge.
- Independent BehaviorSpace replicates conditional on graph seed.

#### 12. Verify assumptions if possible.

| Assumption | Check |
|------------|--------|
| Mean degree $\approx 6$ | `output/lambda_max.csv`, `bs-out-mean-degree` in exports |
| Same $\mathbb{E}[W]$ across laws | `notebooks/recovery_distributions.ipynb`, Figure in report |
| Micro vs. mean-field dynamics consistent | Experiment `08`, `report/fig_er_*` trajectory material |
| Grid censoring | $\hat{\beta}_{\mathrm{surv}\,50}=0.048$ for exponential/lognormal on most seeds → extend $\beta$ grid |
| Graph ensemble | Three seeds `10001`–`10003`; ratios stable $\approx 1.66$–$1.73$ |

## Business/Data Understanding (Get the Data)

Note: automate as much as possible so we can easily get fresh data.

#### 1. List the data we need and how much we need.

| Data | Role | Scale (current repo) |
|------|------|----------------------|
| Edge lists `output/edges/edges_ER_<seed>.csv` | Adjacency for $\lambda_{\max}$ | 3 seeds × ~6000 edges |
| `output/lambda_max.csv` | Spectral reference per network | 3 ER rows (+ optional Lat4/Ring) |
| BehaviorSpace tables `output/raw/*_table.csv` | Per-run metrics | Experiments `05`–`08`; thousands of rows per file |
| Aggregated thresholds | Report tables | `empirical_thresholds.csv`, `threshold_ratio_*.csv` |

**Volume driver:** replicates × $\beta$ grid points × recovery regimes × seeds (see `netlogo/behaviorspace_experiments.xml`).

#### 2. Find and document where we can get that data.

- **Generate:** `python scripts/run_pipeline.py --all` (NetLogo 7 headless + Python aggregation). Requires `NETLOGO_HOME`.
- **Manual path:** NetLogo GUI → BehaviorSpace → import `netlogo/behaviorspace_experiments.xml`.
- **Analysis:** `notebooks/sis_threshold_analysis.ipynb`, `scripts/aggregate_thresholds.py`.

#### 3. Check how much space it will take.

- Edge CSVs: $\mathcal{O}(N \langle k \rangle)$ per seed (~few MB).
- Raw BehaviorSpace exports: tens of MB per large sweep (monitor `output/pipeline_full_run.log`).
- Figures and report PDF: additional ~MB in `report/`.

#### 4. Check legal obligations, and get authorization if necessary.

**Synthetic simulation only**—no human or animal health records. Standard academic use of NetLogo and cited papers.

#### 5. Get access authorizations.

Local install: NetLogo 7, Python 3.x, dependencies in `requirements.txt` / `benchmark/requirements.txt`.

#### 6. Create a workspace (with enough storage space)

Repository root with `output/`, `netlogo/`, `scripts/`, `notebooks/`. Cursor/VS Code kernel **RMAI (.conda)** per project settings.

#### 7. Get the data.

Default pipeline already produced `output/raw/05`–`07` ER baseline tables and `output/empirical_thresholds.csv`. Re-run pipeline after model or XML changes.

#### 8. Convert the data to format we can easily manipulate (without changing the data itself).

- BehaviorSpace → CSV (comma-separated, quoted headers).
- `scripts/analysis_utils.read_behaviorspace_table()` normalises column names.
- NetworkX-style edge list: source/target columns in `output/edges/`.

#### 9. Ensure sensitivity information is deleted or protected (e.g. anonymized).

Not applicable (no personal data).

#### 10. Check the size and type of the data (time series, sample, geographical, etc.)

- **Cross-sectional experiment table:** one row per BehaviorSpace run (replicate).
- **Time series (subset):** experiment `08` exports per-tick prevalence (`bs-out-prev-grp`, `bs-out-prev-mf`).
- **Not geographical:** abstract graph nodes only.

#### 11. Sample a test set, put it aside, and never look at it (no data snooping!)

ML-style hold-out is **not** central here; instead:

- **Hold-out graph seeds:** estimate thresholds on seeds `10002`–`10003` only after fixing analysis code on `10001` (report focuses on `10001` for narrative continuity).
- **Hold-out $\beta$ values:** optional—fit logistic ED50 on a subset of grid points, validate on held-out points (implemented but not required for main conclusions).
- **Do not** tune the NetLogo model to minimise deviation from $\beta_{\mathrm{pred}}$ after seeing survival curves—that would overfit the benchmark.

## Exploratory Analysis (Explore the Data)

Note: try to get insights from a field expert for these steps.

#### 1. Create a copy of the data for exploration (sampling it down to a manageable size if necessary).

Work on copies under `output/`; for quick plots, subsample BehaviorSpace rows or aggregate survival by $(\beta, \text{regime})$ before plotting raw replicates.

#### 2. Create a Jupyter notebook to keep a record of our data exploration.

| Notebook | Focus |
|----------|--------|
| `notebooks/sis_threshold_analysis.ipynb` | Thresholds, survival curves, ratios |
| `notebooks/recovery_distributions.ipynb` | PDFs of recovery laws at fixed $\mathbb{E}[W]$ |
| `notebooks/er_thresholds_summary.ipynb` | ER-focused summaries |
| `notebooks/baseline_empirical_threshold_ER.ipynb` | Baseline RQ1 material |
| `notebooks/virus_dynamics_prevalence.ipynb` | Prevalence / experiment `08` |
| `benchmark/analyze_benchmark_outputs.ipynb` | Python engine cross-check |

This file (`prototype-analysis.ipynb`) is the **project-level** checklist and snapshot.

#### 3. Study each attribute and its characteristics:

| Attribute | Type | Notes |
|-----------|------|--------|
| `infection-prob` ($\beta$) | Float, swept | Design variable; grid step 0.002 near threshold |
| `recovery` chooser | Categorical | exponential / power law (Tang) / lognormal (Tang) |
| `bs-out-extinct` | Binary 0/1 | 1 = absorbed (no infected) |
| `bs-out-final-tick` | Integer | Stopping time; right-censored at `max-ticks` if not extinct |
| `bs-out-late-mean-prevalence` | Float [0,1] | Auxiliary persistence signal |
| `expt-seed` | Integer | Graph draw ID |
| `bs-out-tau-sim` | Float | $\beta \times \mathbb{E}[W]$ logged in model |

**Noise:** intrinsic **stochasticity** of SIS (finite $N$, random recovery draws, random transmission). Not measurement error.

**Distributions:** recovery times heavy-tailed for Tang laws; extinction times right-skewed; survival rate vs. $\beta$ is sigmoid-shaped (motivates logistic ED50).

#### 4. For supervised learning tasks, identify the target attribute(s).

**Outcomes of interest (not class labels from nature):**

- Primary: `bs-out-extinct` (or derived survival rate).
- Derived targets: $\hat{\beta}_{\mathrm{surv}\,50}$, median extinction tick, late prevalence.

#### 5. Visualize the data.

Report figures (`report/fig_*.png`): survival vs. $\beta$, threshold ratio bars, extinction-time violins, recovery PDFs, trajectory micro vs. mean-field. Regenerate via `python scripts/export_report_figures.py`.

#### 6. Study the correlations between attributes.

- Higher $\beta$ → lower extinction probability, higher late prevalence (monotone in $\beta$).
- Same $\beta$, power-law recovery → longer median extinction among extinct runs (see `median_tick_if_extinct` in `empirical_thresholds.csv`).
- $\lambda_{\max}$ weakly varies across ER seeds (~7.18); $\beta_{\mathrm{pred}}$ tracks $1/\lambda_{\max}$.

#### 7. Study how we would solve the problem manually.

See Problem Statement §10; exploratory notebooks implement the manual workflow programmatically.

#### 8. Identify the promising transformations we may want to apply.

- Map $\beta \to \tau = \beta\,\mathbb{E}[W]$ for comparison to $\tau_{\mathrm{pred}}$.
- Aggregate replicates → survival rate per $(\beta, \text{regime}, \text{seed})$.
- Logit transform for ED50 (`threshold_estimators.logistic_ed50_linearized`).
- Bootstrap replicates for CI on $\hat{\beta}_{\mathrm{surv}\,50}$.

#### 9. Identify extra data that would be useful (go back to "Get the Data").

- Extended $\beta$ grid above `0.048` (reduce censoring).
- More ER seeds; lattice/ring experiments `02`–`04` (`--with-lattice-ring`).
- Longer `max-ticks` or quasi-stationary analysis for borderline runs.
- Benchmark CSVs reconciled with NetLogo exports (`benchmark/output/`).

#### 10. Document what we have learned.

Spectral $\beta_{\mathrm{pred}}$ is a **useful but optimistic** control guide for this discrete ABM; heavy-tailed recovery does not uniformly lower the survival threshold; **dynamics** (extinction times) separate regimes more clearly than threshold ratios on the current grid. Details: `report/Further_Exploration_Report.md`.

## Prepare the Data

Notes:
- Work on copies of the data (keep the original dataset intact).
- Write functions for all data transformations we apply, for five reasons:

1. So we can easily prepare the data the next time we get a fresh dataset.
2. So we can apply these transformations in future projects.
3. To clean and prepare the test set.
4. To clean and prepare new data instances once our solution is live.
5. To make it easy to treat our preparation choices as hyperparameters.

#### 1. Data Cleaning:
- Fix or remove outliers (optional).
- Fill in missing values or drop their rows.

**Applied in this project:**

- `read_behaviorspace_table()` strips quotes, normalises headers, drops duplicate column artefacts.
- Runs with missing `bs-out-extinct` are excluded from aggregation (rare export glitches).
- **No outlier removal** on extinction times—long extinctions are scientifically informative for heavy-tailed recovery.
- Flag **censored** threshold estimates when survival $\geq 0.5$ at $\beta_{\max}$ (`RepBetaGridMax` = 0.048).

#### 2. Feature Selection (optioneel):
- Drop the attributes that provide no useful information for the task.

Keep epidemic outputs and design columns; drop unused NetLogo interface metrics. Experiment `01` export tables skip epidemic columns by design.

#### 3. Feature Engineering, where appropriate:
- Discretize continuous features.
- Decompose features (e.g. categorical, date/time, etc.).
- Add promising transformations of features (e.g. $\log(x)$, $\sqrt{x}$, etc.).
- Aggregate features into promising new features.

**Engineered fields (in `scripts/`):**

- `recovery_regime` from filename (`aggregate_thresholds.recovery_regime_from_path`).
- `net_label` from `network-type` / path.
- Survival rate = mean$(1 - \texttt{bs-out-extinct})$ or mean(`extinct < 0.5`) per group.
- $\hat{\beta}_{\mathrm{surv}\,50}$, $\hat{\beta}_{\mathrm{logit\_ed50}}$, ratios to $\beta_{\mathrm{pred}}$, bootstrap CIs (`threshold_estimators.py`).
- $\lambda_{\max}$, $\beta_{\mathrm{pred}}$, heterogeneous MF optional columns in `lambda_max.csv`.

#### 4. Feature Scaling:
- Standardize of normalize features.

**Not used** for threshold estimation ( $\beta$ is already on a natural probability scale). Logit transform only for **ED50 curve fitting**, not for spectral comparison.

## Shortlist Promising Models

*In this project “models” means **simulation + inference approaches**, not sklearn classifiers.*

Notes:
- If the data is huge, we may want to sample smaller training sets so we can train many different models in a reasonable time (be aware that this penalizes complex models such as large neural nets or random Forests). 
- Once again, try to automate these steps as much as possible.

#### 1. Train many quick-and-dirty models from different categories (e.g. linear, naive Bayes, SVM, Random Forest, neural net etc.) using standard parameters.

| Approach | Role | Location |
|----------|------|----------|
| **NetLogo grp-SIS ABM** | Ground-truth stochastic dynamics | `netlogo/virus_simulation.nlogox` |
| **Spectral mean-field** ($c=1$) | Analytic benchmark | `scripts` + `lambda_max.csv` |
| **Heterogeneous MF** (degree moments) | Secondary benchmark | `lambda_max.csv` (`tau_pred_heterogeneous_mf`) |
| **Grid survival threshold** | Nonparametric 50% crossing | `threshold_estimators.survival_curve` |
| **Logistic ED50 on survival** | Smooth threshold if grid sparse | `logistic_ed50_linearized` |
| **Python Numba SIS engine** | Faster exploration / sanity check | `benchmark/sis_numba.py` |

#### 2. Measure and compare their performance.
- For each model, use $N-$ fold cross-validation and compute the mean and standard deviation of the performance measure on the $N$ folds.

**Comparison metrics:**

- $|\hat{\beta}_{\mathrm{surv}\,50} - \beta_{\mathrm{pred}}|$ and ratio $\hat{\beta}/\beta_{\mathrm{pred}}$ (RQ1–RQ2).
- Shift of $\hat{\beta}$ between recovery regimes at fixed seed.
- Distribution of `bs-out-final-tick` (RQ3).
- **Uncertainty:** bootstrap over BehaviorSpace replicates; across seeds `10001`–`10003` for graph variability.

Not $k$-fold CV on agents—replicates are exchangeable Monte Carlo draws.

#### 3. Analyze the most significant variables for each algorithm.

**Drivers of persistence:** $\beta$, $\lambda_{\max}$ (topology), recovery law (tail heaviness), $I_0$, finite $N$, time discretisation.

#### 4. Analyze the types of errors the models make.
- What data would a human have used to avoid these errors?

| “Error” type | Example | Mitigation |
|--------------|---------|------------|
| **Benchmark mismatch** | Spectral line below ABM survival threshold | Report effective $c \approx \hat{\tau}_{50}\lambda_{\max}$; do not claim $c=1$ |
| **Censoring** | Threshold at top of $\beta$ grid | Extend grid; use ED50 |
| **Horizon confusion** | “Survived” at tick 10 000 may still go extinct | Longer runs; absorption-time analysis |
| **Discrete vs. continuous time** | Per-tick $\beta$ vs. $\tau_c$ theory | Emphasise relative shifts between laws |

#### 5. Perform a quick round for feature selection and engineering.

Chose survival-based $\hat{\beta}_{50}$ as primary; kept legacy late-prevalence persistence and logit ED50 as sensitivity columns in `empirical_thresholds.csv`.

#### 6. Perform one or two more quick iterations of the five previous steps.

Iteration 1: single seed `10001`. Iteration 2: three seeds + bootstrap CIs + report figures. Optional: `benchmark/` parallel engine compared to NetLogo exports.

#### 7. Shortlist the top three to five most promising models, preferring models that make different types of errors.

**Shortlist for reporting:**

1. **NetLogo ABM + grid $\hat{\beta}_{\mathrm{surv}\,50}$** — primary evidence.
2. **Spectral $\beta_{\mathrm{pred}}$** — theory benchmark (RQ1).
3. **Extinction-time summaries** — best discriminator for RQ3.
4. **Logistic ED50** — when grid is coarse or censored.
5. **Python benchmark engine** — rapid parameter scans (confirm before citing in thesis).

## Fine-Tune the System

Notes:
- We will want to use as much data as possible for this step, especially as we move toward the end of fine-tuning.
- Automatiseer zoveel als mogelijk.

#### 1. Fine-tune the hyperparameters using cross-validation:
- Treat your data transformation choices as hyperparameters, especially when we are not sure about them (e.g. if you are not sure whether to replace missing values with zeros or with the medo=ian value, or to just drop te rows).
- Unless there are very few hyperparameter values to explore, prefer random search over grid search. If training is very long, we may prefer a Bayesian optimization approach.

**Simulation / analysis hyperparameters tuned or documented:**

| Hyperparameter | Choice | Rationale |
|----------------|--------|-----------|
| `num-nodes`, `avg-degree` | 2000, 6 | Proposal fixed design |
| `recovery-mean` | 5 | Match $\mathbb{E}[W]$ across laws |
| Tang `power-law-lambda` | 4.24 | Literature / Tang implementation |
| Tang `lognormal-sigma` | 1 | Heavy tail at fixed mean |
| `initial-infected` | 5 | Small seed outbreak |
| `max-ticks` | 10 000 | Trade-off runtime vs. slow extinction |
| $\beta$ grid | ~0.014–0.048, step 0.002 | Coarse-to-fine around $\beta_{\mathrm{pred}}$ |
| BehaviorSpace `repetitions` | 24 | Stabilise survival estimates |
| Threshold rule | 50% survival vs. extinct | Operational endemic proxy |
| Estimator | Grid vs. logit ED50 | Sensitivity when censored |

**Not tuned to fit theory:** recovery-law parameters were fixed a priori from Tang et al., not optimised to minimise $|\hat{\beta}-\beta_{\mathrm{pred}}|$.

#### 2. Try ensemble methods.. Combining our best models will often produce better performance than ruinning them individually.

**Ensemble in the epidemiology sense:**

- Triangulate **ABM thresholds** with **spectral** and optional **heterogeneous MF** lines.
- Combine **three graph seeds** for robustness (mean/std in `threshold_ratio_summary.csv`).
- Do **not** average NetLogo and benchmark engine outputs without reconciling time discretisation and random number streams.

#### 3. Once we are confident about our final model, measure its performance on the test set to estimate the generalization error.

**Generalisation read as:**

- **Across seeds:** ratios stable (exponential mean ratio $\approx 1.73$, PL $\approx 1.66$, lognormal $\approx 1.70$).
- **Across recovery laws:** partial qualification of heavy-tail hypothesis.
- **Not claimed:** transfer to empirical contact networks or different $N$, $\langle k\rangle$, or $I_0$ without new experiments.

## Present Our Solution

#### 1. Document what we have done.

- **Report:** `report/report.tex` → `report/report.pdf` (methods, RQ1–RQ3, figures).
- **Proposal:** `proposal/research_proposal.pdf`.
- **Pipeline:** `scripts/run_pipeline.py`, `scripts/build_full_report.py`, `scripts/write_report_macros.py`.
- **Exploration memo:** `report/Further_Exploration_Report.md`.
- **This notebook:** prototype-phase checklist and snapshot metrics.

#### 2. Create a nice presentation.
- Make ure we highlight the big picture first.

**Suggested narrative order:**

1. Problem: when does SIS persist on a network?
2. Benchmark: $\beta_{\mathrm{pred}} = 1/(\lambda_{\max}\mathbb{E}[W])$.
3. Method: NetLogo ABM + exported edges + Python aggregation.
4. Result: empirical threshold **above** prediction; heavy tails matter more for **timing** than a uniform threshold drop.
5. Limitations: censoring, ER topology, finite horizon.

Figures ready in `report/fig_*.png`.

#### 3. Explain why our solution achieves the business objective.

We deliver **evidence-backed guidance**: the spectral line is a **scale** and **conservative** control reference for this implementation, not a sharp critical $\beta$. Practitioners comparing recovery laws should inspect **extinction-time distributions**, not only a single threshold ratio.

#### 4. Don't forget to present interesting points we noticed along the way.
- Describe what worked and what did not.
- List our assumtions and our system's limitations.

**Worked well:** reproducible headless pipeline; three seeds; bootstrap CIs; trajectory check (micro ≈ homogeneous MF when defined consistently).

**Did not fully work as hypothesised:** heavy tails did **not** uniformly lower $\hat{\beta}_{\mathrm{surv}\,50}$; lognormal often censored like exponential.

**Limitations:** $\beta$ grid cap; discrete-time vs. continuous theory; ER ≠ realistic contacts; survival ≠ true endemicity at 10 000 ticks.

#### 5. Ensure our key findings are comunicated through beautiful visualizations or easy-to-remeber statements.

> **“Spectral $\beta_{\mathrm{pred}}$ is about 40–45% below the $\beta$ where half of our runs still show infection at the time horizon.”**

> **“Change the recovery *law*, and extinction times move more than the 50% survival threshold on this grid.”**

Regenerate plots: `python scripts/export_report_figures.py`.

## Launch!

*For a research artefact, “launch” means **reproducible release**, not production deployment.*

#### 1. Get our solution ready for production (plug into production data inputs, write unit tests, etc.).

**Reproducibility checklist:**

- [ ] Pin NetLogo version (7.0.3 in model).
- [ ] Document `NETLOGO_HOME` and Python env (`requirements.txt`).
- [ ] Run `python scripts/run_pipeline.py --all` on a clean clone.
- [ ] Build report: `python scripts/build_full_report.py` (or documented LaTeX command).
- [ ] Commit `output/` CSVs or provide download / Zenodo archive (per university policy).
- [ ] Optional: smoke-test `benchmark/run_benchmark.py` against one NetLogo export.

No unit tests are mandatory in-repo today; adding tests for `threshold_estimators` and `read_behaviorspace_table` would harden the pipeline.

#### 2. Write monitoring code to check our system's live performance at regular intervals and trigger alerts when it drops.
- Beware of slow degradation: models tend to "rot" as data evolves.
- Measuring performance may require a human pipeline (e.g. via a crowdsourcing service).
- Also monitor your inputs quality. This is particularly important for online learning systems.

**Research analogue:**

- Log BehaviorSpace run failures and row counts after each pipeline stage (`output/pipeline_full_run.log`).
- Alert if $\hat{\beta}_{\mathrm{surv}\,50} = \beta_{\max}$ for all regimes → **censoring warning**.
- Compare new `empirical_thresholds.csv` to archived baseline when re-running after model edits.

#### 3. Retrain our models on a regular basis on fresh data (automate as much as possible).

**Re-run when:** NetLogo model logic changes, experiment XML changes, new graph seeds, or extended $\beta$ grid.

Automate via CI or a Makefile target invoking `run_pipeline.py --all` and `aggregate_thresholds` (long-running; suitable for batch HPC or overnight job).

## Conclusions and Recommendations

### Conclusions

1. **RQ1:** On ER graphs with exponential recovery, the operational survival threshold $\hat{\beta}_{\mathrm{surv}\,50}$ lies systematically **above** the mean-field spectral prediction $\beta_{\mathrm{pred}}$ (effective $c \approx 1.6$–$1.7$ in $\tau$-space), so $c=1$ is a useful **conservative** guide, not a sharp critical value for this discrete-time ABM.

2. **RQ2:** Tang power-law and lognormal recovery at fixed $\mathbb{E}[W]=5$ produce **modest** shifts in $\hat{\beta}_{\mathrm{surv}\,50}$ relative to exponential recovery and relative to the gap vs. $\beta_{\mathrm{pred}}$; several estimates are **censored** at the largest tested $\beta$.

3. **RQ3:** Recovery law changes **extinction-time** behaviour more clearly than a single “heavy tail lowers threshold” story (e.g. longer median extinction among extinct runs under power-law recovery).

4. The **original hypothesis** is **partly qualified**: heavy-tailed recovery does not uniformly ease persistence at unchanged mean infectious period on this design.

### Recommendations (next work)

| Priority | Action |
|----------|--------|
| High | Extend $\beta$ grid above `0.048`; refine near $\beta_{\mathrm{pred}}$ |
| High | Use logistic ED50 + report bootstrap CIs wherever grid is censored |
| Medium | Stratify extinction times by $\beta$ (full RQ3) |
| Medium | Optional lattice/ring runs (`--with-lattice-ring`) |
| Lower | Reconcile `benchmark/output/` with NetLogo exports before citing both |

## System of Algorithms

End-to-end flow (reproducible research pipeline):

```mermaid
flowchart LR
  subgraph NetLogo
    A[virus_simulation.nlogox] --> B[BehaviorSpace sweeps]
    B --> C[output/raw/*_table.csv]
    A --> D[output/edges/*.csv]
  end
  subgraph Python
    D --> E[lambda_max + beta_pred]
    C --> F[aggregate_thresholds]
    E --> F
    F --> G[empirical_thresholds.csv]
    G --> H[export_report_figures]
    H --> I[report/report.pdf]
  end
```

**Key scripts:**

| Step | Module |
|------|--------|
| Headless runs | `scripts/run_pipeline.py` |
| Read CSV / labels | `scripts/analysis_utils.py` |
| Thresholds & CIs | `scripts/threshold_estimators.py` |
| Aggregation | `scripts/aggregate_thresholds.py` |
| LaTeX macros | `scripts/write_report_macros.py` |
| Fast probe (optional) | `benchmark/run_benchmark.py` |

## Final Remarks

This project is a **computational science** study framed with the ML prototype checklist: the “product” is justified epidemiological insight and reproducible simulation artefacts, not a deployed classifier.

The most important methodological lesson for similar work: **separate** (i) agreement between micro-dynamics and mean-field equations on trajectories from (ii) agreement between a **stochastic finite-horizon survival rule** and a **spectral threshold line**. This repository implements both checks.

For hands-on analysis beyond this checklist, open `notebooks/sis_threshold_analysis.ipynb` and run the code cell below to load current aggregated results.

In [2]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
for p in [ROOT, *ROOT.parents]:
    if (p / "output" / "empirical_thresholds.csv").is_file():
        ROOT = p
        break

emp = pd.read_csv(ROOT / "output" / "empirical_thresholds.csv")
ratio = pd.read_csv(ROOT / "output" / "threshold_ratio_summary.csv")
lam = pd.read_csv(ROOT / "output" / "lambda_max.csv")

print("Project root:", ROOT)
print("\n=== threshold_ratio_summary ===")
display(ratio)
print("\n=== empirical_thresholds (ER) ===")
display(emp[emp["net_label"] == "ER"])
print("\n=== lambda_max (ER seeds) ===")
display(lam[lam["net_label"].astype(str).str.contains("ER", na=False)])

Project root: C:\Dossiers\master-rmai-researchproject-virussimulation

=== threshold_ratio_summary ===


,recovery_regime,net_label,ratio_mean,ratio_std,n_seeds
0,exponential,ER,1.727671,0.008704,3
1,lognormal (Tang),ER,1.703755,0.046911,3
2,power law (Tang),ER,1.655884,0.079128,3



=== empirical_thresholds (ER) ===


,recovery_regime,net_label,random_seed,beta_pred,beta_hat_surv_50,beta_hat_surv_25,beta_hat_surv_50_median_boot,beta_hat_surv_50_ci_low,beta_hat_surv_50_ci_high,beta_hat_logit_ed50_surv,beta_hat_persist_50,beta_hat_persist_relaxed_50,median_tick_if_extinct,median_tick_if_survive,ratio_surv_50_over_pred,beta_hat_50_legacy_persist,ratio_legacy_over_pred
0,exponential,ER,10001,0.027853,0.048,0.048,0.048,0.048,0.048,NaN,0.048,0.048,8.0,NaN,1.723351,0.048,1.723351
1,exponential,ER,10002,0.027623,0.048,0.048,0.048,0.048,0.048,NaN,0.048,0.048,12.5,NaN,1.737690,0.048,1.737690
2,exponential,ER,10003,0.027875,0.048,0.048,0.048,0.048,0.048,NaN,0.048,0.048,9.0,NaN,1.721972,0.048,1.721972
3,lognormal (Tang),ER,10001,0.027853,0.048,0.048,0.048,0.048,0.048,NaN,0.048,0.048,9.0,NaN,1.723351,0.048,1.723351
4,lognormal (Tang),ER,10002,0.027623,0.048,0.048,0.048,0.048,0.048,NaN,0.048,0.048,10.5,NaN,1.737690,0.048,1.737690
5,lognormal (Tang),ER,10003,0.027875,0.046,0.046,0.046,0.046,0.046,NaN,0.046,0.046,8.0,10000.0,1.650223,0.046,1.650223
6,power law (Tang),ER,10001,0.027853,0.044,0.044,0.044,0.044,0.044,NaN,0.048,0.048,15.0,10000.0,1.579738,0.048,1.723351
7,power law (Tang),ER,10002,0.027623,0.048,0.048,0.048,0.048,0.048,NaN,0.048,0.048,13.0,NaN,1.737690,0.048,1.737690
8,power law (Tang),ER,10003,0.027875,0.046,0.046,0.046,0.046,0.046,NaN,0.048,0.048,33.5,10000.0,1.650223,0.048,1.721972



=== lambda_max (ER seeds) ===


,edge_file,net_label,random_seed,num_nodes,num_edges,k_mean,k2_mean,tau_pred_homogeneous_mf,tau_pred_heterogeneous_mf,lambda_max
0,C:\Dossiers\Master AI\Research Methods for AI\...,ER,10001,2000,5979,5.979,41.862,0.167252,0.142826,7.180628
1,C:\Dossiers\Master AI\Research Methods for AI\...,ER,10002,2000,6083,6.083,42.981,0.164393,0.141528,7.240375
2,C:\Dossiers\Master AI\Research Methods for AI\...,ER,10003,2000,5979,5.979,41.963,0.167252,0.142483,7.174885


## References

### Project documents

- `proposal/research_proposal.pdf` — RQ1–RQ3, methods, schedule.
- `report/report.pdf` — results, notation, discussion, limitations.
- `report/Further_Exploration_Report.md` — follow-up agenda.
- `references/Tang_Yao_Xie_Feng_grp-SIS_samenvatting.md` — grp-SIS summary.

### Literature (see `report/report.tex` bibliography)

- Erdős & Rényi (1959); Chakrabarti et al. (2008); Van Mieghem (2011); Pastor-Satorras et al. (2015); Cator et al. (2013); Tang et al. (2025).

### Software

- [NetLogo 7](https://ccl.northwestern.edu/netlogo/)
- [Hands-On Machine Learning](https://github.com/ageron/handson-ml3) — Ch. 2 checklist template for this notebook